# Dev Notebook: Continuous Load Increase

## Imports and Definitions

In [ ]:
# %matplotlib inline
# import ipympl

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps as cm

parent = os.path.abspath("/Users/maxikoehler/Documents/GitHub/diffpssi-ma-kohler/")
sys.path.insert(1, parent)
sys.path.append(
    str(
        os.path.dirname(
            os.path.dirname(os.path.abspath("continuous_load_increase.ipynb"))
        )
    )
)
save = os.path.join(os.getcwd(), "plots/")

import matplotlib as mpl
from tools import *

In [ ]:
# import load models
import development_files.examples.ibb_transformer.ibb_trans_model as ibb
from development_files.examples.simple_load.simple_load_mod import load as simple_load


from src.diffpssi import PowerSystemSimulation as Pss
from src.diffpssi import Recorder

# from src.diffpssi.stability_lib import VoltageStability
from src.diffpssi.stability_lib.voltage import NoseCurve

## 1 Set Up Grid, Recorder, and Sim 

In [ ]:
def record_dict_smload(simulation, call=False):
    record_dict = {
        "B0: V": simulation.busses[0].get_value("voltage_mag"),
        "B1: V": simulation.busses[1].get_value("voltage_mag"),
        "B0: S": np.abs(simulation.busses[0].get_value("S")),
        "B1: S": np.abs(
            simulation.busses[1].get_value("S")
        ),  # models[0].i_inj / simulation.base_voltage),
        # 'Load 0: current injection':    np.abs(simulation.busses[1].models[0].get_current_injections()),
    }
    if call:
        return record_dict.values()
    else:
        return record_dict

In [ ]:
def load_dependency(t, start_time=1, initial_p=400):
    """Generating a time dependency for the load in the simulation.

    :param t: Simulation timestep.
    :type t: _type_
    """
    p = initial_p + t * 80
    return p

In [ ]:
param_dict_oltc = {
    "t_1": 5,
    "db": 0.05,
    "delta_m": 0.02,
    "m_max": 1.1,
    "m_min": 0.9,
    "v_ref": 1,
}

param_dict_fsm = {
    "t_m": 0.02,
    "t_k": 5,
    "db": 0.025,
    "delta_m": 2,
    "delta_k": 0.02,
    "m_max": 4,
    "m_min": -4,
    "k_max": 10,
    "k_min": -10,
    "v_ref": 1,
    "gamma_max": 8,
    "pt_1": 0.01,
}

sim = Pss(
    parallel_sims=1,
    sim_time=120,
    time_step=0.005,
    solver="heun",
    grid_data=simple_load(
        trans_type="simple",
        trans_control="oltc",
        param_dict_oltc=param_dict_oltc,
        controllers=False,
        machine_size=2200,
    ),
)

# sim.trafos[0].oltc.dir = -1
sim.add_param_dependency(1, sim.busses[1].models[0], "p_soll_mw", load_dependency)

rec = Recorder(sim=sim, recorder_dict=record_dict_smload)

sim.set_record_function(rec.record_fun)
record_list = rec.record_list()

t, recorder = sim.run()

In [ ]:
sim_results = pd.DataFrame(recorder[0, :, :], columns=record_list)
sim_results["time"] = np.round(t, 3)
sim_results.set_index("time", inplace=True)

sim_results.head()

print(sim_results)

## 2 Short Look into Simulation Behavior

In [ ]:
plt.figure()
plt.plot(sim_results.index, sim_results["B0: V"], label="B0: V")
plt.plot(sim_results.index, sim_results["B1: V"], label="B1: V")

plt.grid()
plt.legend()
plt.xlabel("Time in [s]")
plt.ylabel("Voltage in [p.u]")

plt.show()

In [ ]:
plt.figure()
plt.plot(sim_results.index, sim_results["B0: S"] * 100, label="B0: S")
plt.plot(sim_results.index, sim_results["B1: S"], label="B1: S")

plt.plot(sim_results.index, np.real(sim_results["B0: S"]) * 100, label="B0: P")
plt.plot(sim_results.index, np.real(sim_results["B1: S"]), label="B1: P")

plt.plot(sim_results.index, np.imag(sim_results["B0: S"]) * 100, label="B0: Q")
plt.plot(sim_results.index, np.imag(sim_results["B1: S"]), label="B1: Q")

plt.grid()
plt.legend()
plt.xlabel("Time in [s]")
plt.ylabel("Power in [MVA]")

plt.show()

## 3 Plotting the TDS Solution into the Nose Curves

In [ ]:
p_load = np.linspace(0, 10000, 1000)
tan_phi = [0]  # np.linspace(-0.3, 0.3, 3)

load_dict = {
    "p": p_load,
    "tan_phi": tan_phi,
}
sim.verbose = False

nc = NoseCurve(load_model=simple_load, loading=load_dict, ps_sim=sim)

res_nc = nc.run_calculation(["B1"])["B1"]

In [ ]:
cmap = plt.get_cmap("bone")

load_model = {
    "z": 1,
    "i": 0,
    "p": 0,
}

filter_ = 0.5

p = np.real(sim_results.loc[sim_results.index % filter_ == 0]["B1: S"])
v = sim_results.loc[sim_results.index % filter_ == 0]["B1: V"]

ax = nc.plot_nose_curve(["B1"], size=(6, 4), title=False)
ax.scatter(
    p,
    v,
    marker=".",
    c=sim_results.loc[sim_results.index % filter_ == 0].index,
    cmap=cmap,
    label="Time Development Simulation",
)
# ax.plot(p, v, label='Time Development Simulation')
# ax = nc.add_load_to_plot(load=[400,0], load_model=load_model, bus='B1', current_plot=ax, y_lims=[0.2, 1.1])
# ax = nc.add_load_to_plot(load=[5000,0], load_model=load_model, bus='B1', current_plot=ax, y_lims=[0.2, 1.1])

# ax.legend(loc='lower right')
ax.legend()

# plt.savefig('./plots/nose_curve_tds_load.pdf')
plt.savefig("./plots/nose_cuve_small-machine.pdf")
plt.show()

## Machine without Controllers

### 2200 MVA

In [ ]:
sim_2 = Pss(
    parallel_sims=1,
    sim_time=120,
    time_step=0.005,
    solver="heun",
    grid_data=simple_load(
        trans_type="simple",
        trans_control="oltc",
        param_dict_oltc=param_dict_oltc,
        controllers=True,
        machine_size=5 * 2200,
    ),
)

# sim_2.trafos[0].oltc.dir = -1
sim_2.add_param_dependency(1, sim_2.busses[1].models[0], "p_soll_mw", load_dependency)

rec_2 = Recorder(sim=sim_2, recorder_dict=record_dict_smload)

sim_2.set_record_function(rec_2.record_fun)
record_list = rec_2.record_list()

t, recorder_woc_2200 = sim.run()

In [ ]:
print(sim_2.param_events[0].value(50))

In [ ]:
sim_results_2 = pd.DataFrame(recorder_woc_2200[0, :, :], columns=record_list)
sim_results_2["time"] = np.round(t, 3)
sim_results_2.set_index("time", inplace=True)

sim_results_2.head()

In [ ]:
p_load = np.linspace(0, 10000, 1000)
tan_phi = [0]  # np.linspace(-0.3, 0.3, 3)

load_dict = {
    "p": p_load,
    "tan_phi": tan_phi,
}
sim_2.verbose = False

nc_2 = NoseCurve(load_model=simple_load, loading=load_dict, ps_sim=sim_2)

res_nc_2 = nc_2.run_calculation(["B1"])["B1"]

cmap = plt.get_cmap("bone")

filter_ = 0.5

p_2 = sim_results_2.loc[sim_results_2.index % filter_ == 0]["B1: S"]
v_2 = sim_results_2.loc[sim_results_2.index % filter_ == 0]["B1: V"]

print(p_2)

ax_2 = nc_2.plot_nose_curve(["B1"], size=(6, 4), title=False)
ax_2.scatter(
    p_2,
    v_2,
    marker=".",
    c=sim_results_2.loc[sim_results_2.index % filter_ == 0].index,
    cmap=cmap,
    label="Time Development Simulation",
)

# ax.legend(loc='lower right')
ax_2.legend()

plt.savefig("./plots/nose_cuve_small-machine.pdf")
plt.show()

### 5 * 2200 MVA